In [8]:
import pickle
from pathlib import Path

# Main AI 311 project directory
project_dir = Path(
    r"C:\Users\dvdsl\OneDrive\Documents\GitHub\AI 311"
)

model_path = project_dir / "Data/processed/final_phishing_model.pkl"

# Load your existing final model
with open(model_path, "rb") as f:
    final_model = pickle.load(f)

print("Model loaded successfully!")
print("Model type:", type(final_model))
print("Model parameters:")
print(final_model.get_params())

Model loaded successfully!
Model type: <class 'sklearn.svm._classes.SVC'>
Model parameters:
{'C': 1, 'break_ties': False, 'cache_size': 200, 'class_weight': None, 'coef0': 0.0, 'decision_function_shape': 'ovr', 'degree': 3, 'gamma': 'scale', 'kernel': 'linear', 'max_iter': -1, 'probability': False, 'random_state': None, 'shrinking': True, 'tol': 0.001, 'verbose': False}


In [9]:
with open(project_dir / "Data/processed/final_phishing_model.pkl", "rb") as f:
    model = pickle.load(f)

with open(project_dir / "Data/processed/processed_data.pkl", "rb") as f:
    X_train, X_test, train_labels, test_labels, vectorizer = pickle.load(f)

print("Model:", model)
print("Test data:", X_test.shape)

Model: SVC(C=1, kernel='linear')
Test data: (16416, 5000)


In [10]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    balanced_accuracy_score,
    matthews_corrcoef
)

# Predict using the final SVM
predictions = model.predict(X_test)

# Get confusion matrix values
tn, fp, fn, tp = confusion_matrix(
    test_labels, predictions, labels=[0, 1]
).ravel()

print("========== SVM MODEL PERFORMANCE ==========")

print(f"Accuracy: {accuracy_score(test_labels, predictions):.4f}")
print(f"Precision: {precision_score(test_labels, predictions):.4f}")
print(f"Recall: {recall_score(test_labels, predictions):.4f}")
print(f"F1 Score: {f1_score(test_labels, predictions):.4f}")

print(f"Specificity: {tn / (tn + fp):.4f}")
print(f"False Positive Rate: {fp / (fp + tn):.4f}")
print(f"False Negative Rate: {fn / (fn + tp):.4f}")

print(f"Balanced Accuracy: {balanced_accuracy_score(test_labels, predictions):.4f}")
print(f"MCC: {matthews_corrcoef(test_labels, predictions):.4f}")

print("\n========== CONFUSION MATRIX ==========")
print(confusion_matrix(test_labels, predictions))

print("\n========== CLASSIFICATION REPORT ==========")
print(classification_report(test_labels, predictions))

========== SVM MODEL PERFORMANCE ==========
Accuracy: 0.9848
Precision: 0.9855
Recall: 0.9851
F1 Score: 0.9853
Specificity: 0.9845
False Positive Rate: 0.0155
False Negative Rate: 0.0149
Balanced Accuracy: 0.9848
MCC: 0.9695

========== CONFUSION MATRIX ==========
[[7787  123]
 [ 127 8379]]

========== CLASSIFICATION REPORT ==========
              precision    recall  f1-score   support

           0       0.98      0.98      0.98      7910
           1       0.99      0.99      0.99      8506

    accuracy                           0.98     16416
   macro avg       0.98      0.98      0.98     16416
weighted avg       0.98      0.98      0.98     16416



In [11]:
import pandas as pd
import numpy as np

# Build a dataframe of the predictions
results = pd.DataFrame({
    "Actual": np.asarray(test_labels),
    "Predicted": predictions,
    "Decision_Score": final_model.decision_function(X_test)
}, index=test_labels.index)

# Identify missed phishing emails
false_negatives = results[
    (results["Actual"] == 1) &
    (results["Predicted"] == 0)
]

# Identify legitimate emails flagged as phishing
false_positives = results[
    (results["Actual"] == 0) &
    (results["Predicted"] == 1)
]

print("False Negatives:", len(false_negatives))
print("False Positives:", len(false_positives))

print("\nFirst 10 missed phishing records:")
print(false_negatives.head(10))

False Negatives: 127
False Positives: 123

First 10 missed phishing records:
       Actual  Predicted  Decision_Score
81005       1          0       -1.166525
18139       1          0       -0.143166
20565       1          0       -2.158711
81929       1          0       -0.281136
17320       1          0       -0.196273
3548        1          0       -0.753200
18623       1          0       -0.340714
18463       1          0       -2.146352
26540       1          0       -0.092459
17976       1          0       -0.027220
